## Árvore de Decisão

A [Árvore de Decisão](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) é um modelo baseado em regras, que realiza divisões sucessivas nos dados com o objetivo de separar as classes.

Cada divisão é feita com base em uma variável e um ponto de corte, formando uma estrutura semelhante a uma árvore.

Esse modelo é interpretável e capaz de capturar relações não lineares entre as variáveis.

### Hiperparâmetros utilizados

- **max_depth**: define a profundidade máxima da árvore.
  - Valores menores → modelo mais simples (menos overfitting)
  - Valor `None` → árvore cresce livremente

- **min_samples_split**: número mínimo de amostras necessárias para dividir um nó.
  - Valores maiores tornam o modelo mais conservador

- **min_samples_leaf**: número mínimo de amostras em uma folha.
  - Evita que a árvore crie divisões muito específicas

- **criterion**: métrica utilizada para avaliar a qualidade da divisão.
  - `gini`: índice de Gini (default)
  - `entropy`: ganho de informação

### Estratégia

Foi utilizado o GridSearchCV com validação cruzada (5 folds) para ajustar os hiperparâmetros e reduzir o risco de overfitting.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

from preprocessing import load_and_preprocess

X_train, X_test, y_train, y_test = load_and_preprocess(
    "../predictive_models/scrdata_202505.csv"
)

param_grid_dt = {
    "max_depth": [30, 35, 40],
    "min_samples_split": [2, 3],
    "min_samples_leaf": [15, 25, 45, 65],
    "class_weight": [None, "balanced"],
    "criterion": ["gini", "entropy"]
}

grid_dt = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid_dt,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_dt.fit(X_train, y_train)

print("Melhores parâmetros:", grid_dt.best_params_)
print("Melhor score (CV):", grid_dt.best_score_)

best_dt = grid_dt.best_estimator_

y_pred = best_dt.predict(X_test)
y_proba = best_dt.predict_proba(X_test)[:, 1]

roc = roc_auc_score(y_test, y_proba)
f1 = f1_score(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)

resultados = {
    "modelo": "DecisionTree",
    "roc_auc": roc,
    "f1": f1,
    "accuracy": acc,
    "roc_auc_cv": grid_dt.best_score_,
    "best_params": str(grid_dt.best_params_)
}

df_result = pd.DataFrame([resultados])
df_result.to_csv("resultados_dt.csv", index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'predictive_models/scrdata_202505.csv'